<style>.hero{padding:40px 44px;border-radius:24px;color:#f8fbff;background:linear-gradient(125deg,#102a43,#184e77 55%,#2a9d8f);box-shadow:0 16px 38px #102a433d;font-family:Inter,'Segoe UI',sans-serif}.hero h1{font-size:38px;color:white;margin:12px 0}.chips span{display:inline-block;padding:6px 13px;margin:3px;border:1px solid #ffffff55;border-radius:999px;background:#ffffff20;font-size:12px;font-weight:800}.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:12px}.card{padding:15px;border-radius:12px;background:#ffffff16;border:1px solid #ffffff38}.section{margin:28px 0 14px;padding:14px 20px;border-radius:13px;background:linear-gradient(90deg,#102a43,#184e77);color:white}.call{margin:17px 0;padding:20px 23px;border:1px solid #c9d9e6;border-left:6px solid #2a9d8f;border-radius:13px;background:#edf7f6;color:#172033}.warn{border-left-color:#e9a23b;background:#fff8e8}.kpi{display:inline-block;min-width:155px;margin:5px;padding:13px;border-radius:11px;background:#edf5fb;border:1px solid #c9d9e6;text-align:center}.kpi b{display:block;font-size:22px;color:#184e77}table{width:100%}th{background:#184e77!important;color:white!important;text-align:left!important}td,th{padding:9px!important}</style><div class="hero"><div class="chips"><span>DEEP LEARNING</span><span>PROYECTO 1</span><span>V3 PROMOVIDA</span><span>GRUPO 1 · SECCIÓN 30</span></div><h1>Monitoreo transaccional y detección de fraude</h1><p style="font-size:20px">LightGBM nativo, recencia, calibración y decisión económica</p><div class="grid"><div class="card"><b>Institución</b><br>Universidad del Valle de Guatemala</div><div class="card"><b>Curso</b><br>Deep Learning y Sistemas Inteligentes</div><div class="card"><b>Docente</b><br>Kevin Recinos</div><div class="card"><b>Integrantes</b><br>Wilson Calderón · 22018<br>Pablo Barillas · 22193</div><div class="card"><b>Dataset</b><br>IEEE-CIS Fraud Detection</div><div class="card"><b>Candidato</b><br>LGB native recency V3</div></div></div>

In [1]:
from pathlib import Path
import json, pandas as pd
from IPython.display import Image, display
ROOT=Path.cwd()
if not (ROOT/'artefactos').exists(): ROOT=ROOT.parents[1]
R=json.loads((ROOT/'artefactos/v3/resultados_v3.json').read_text(encoding='utf-8'))
R['recomendacion']

'Promover V3 con umbral balanceado'

<div class="section"><h2>1 · Resumen ejecutivo</h2></div>

<div class="call"><b>Decisión:</b> V3 se promueve porque cumplió los cuatro criterios definidos antes de consultar el benchmark. Ganó 3/3 folds, elevó AP walk-forward en 0.1082 y redujo costo holdout 18.77%.</div>

El objetivo no fue aumentar complejidad indiscriminadamente, sino mejorar representación, protocolo y punto operativo. AUC-PR evalúa ranking bajo desbalance; el costo $$C(\tau)=4200FN(\tau)+180FP(\tau)$$ expresa el supuesto económico académico.

In [2]:
pd.DataFrame(R['promocion']['criterios'],index=['cumple']).T

,cumple
auc_pr_walk_delta_min_0_015,True
costo_holdout_reduccion_min_3pct,True
recall_no_cae_mas_1pp,True
gana_al_menos_2_de_3_folds,True


<div class="section"><h2>2 · Datos y causalidad</h2></div>

Los 590,540 eventos se ordenan por `TransactionDT`. `TransactionID` une tablas, pero ninguno entra como predictor continuo. Para cada evento, $$x_t^{hist}=f(\{x_j:t_j<t\}).$$ Los agregados de monto, recencia y frecuencia se emiten antes de incorporar la transacción actual.

In [3]:
pd.DataFrame(R['datos']['particiones']).T

,n,fraude,dt_min,dt_max
audit_train,324797.0,0.033870,86400.0,8022306.0
train,413378.0,0.035169,86400.0,10437996.0
validation,88581.0,0.034341,10438003.0,13151840.0
benchmark_historico,88581.0,0.034804,13151880.0,15811131.0


<div class="section"><h2>3 · Variables, correlación y categorías</h2></div>

La selección se aprende en el 55% inicial. Se retienen 220 numéricas y 24 categóricas; solo se eliminan redundancias casi monotónicas con $|\rho_s|\ge0.995$. LightGBM recibe categorías nativas y conserva `NaN`, evitando orden artificial e imputación que borre patrones de ausencia.

In [4]:
pd.DataFrame({'tipo':['numéricas','categóricas'],'cantidad':[R['seleccion_variables']['n_numericas'],R['seleccion_variables']['n_categoricas']]})

,tipo,cantidad
0,numéricas,220
1,categóricas,24


<div class="section"><h2>4 · Walk-forward</h2></div>

Tres futuros simulados comparan entrenamiento uniforme, ponderación por recencia y las 300 mil observaciones más recientes. La recencia gana los tres folds; la evidencia sugiere deriva gradual y utilidad parcial del pasado antiguo.

In [5]:
pd.read_csv(ROOT/'artefactos/v3/validacion_walk_forward_v3.csv').pivot(index='fold',columns='modelo',values='auc_pr')

modelo,LGB_native_recency,LGB_native_recent300k,LGB_native_uniform
fold,,,
F1,0.567967,0.558759,0.565006
F2,0.622546,0.609277,0.616044
F3,0.552214,0.535729,0.539349


<div class="section"><h2>5 · Regresión logística y PCA</h2></div>

L2, L1 y Elastic Net corroboran una línea base lineal. Alcanzaron el máximo de iteraciones y se etiquetan como no convergidos. PCA64 conserva 99.53% de varianza, pero reduce AUC-PR: varianza total y señal de fraude no son equivalentes.

In [6]:
pd.read_csv(ROOT/'artefactos/v3/baselines_logisticos_v3.csv')

,modelo,auc_pr_validacion,roc_auc_validacion,segundos,iteraciones,variables_transformadas,varianza_pca
0,Logistica_ElasticNet,0.331162,0.763164,191.840145,100,274,NaN
1,Logistica_L2,0.331161,0.763146,140.791437,100,274,NaN
2,Logistica_L1,0.331004,0.763153,204.596034,100,274,NaN
3,Logistica_PCA64,0.303202,0.723722,1.339713,38,64,0.995282


<div class="section"><h2>6 · Calibración y umbrales</h2></div>

Early stopping, calibración y umbral usan bloques sucesivos. El umbral balanceado maximiza F1 con recall mínimo de 70%; el económico minimiza costo. La calibración reduce Brier y ECE sin alterar el orden de los puntajes.

In [7]:
pd.DataFrame(R['modelo_v3']['calibracion'],index=['valor']).T

,valor
brier_raw_validacion,0.020642
brier_calibrado_validacion,0.020550
ece_raw_validacion,0.006824
ece_calibrado_validacion,0.004666


<div class="section"><h2>7 · Resultados V1/V2/V3</h2></div>

V3 mejora simultáneamente AUC-PR, precisión, recall, F1 y costo frente a V2 usando el umbral balanceado. El benchmark es histórico reutilizado, de modo que la tabla es evidencia descriptiva y no confirmación externa.

In [8]:
pd.DataFrame(R['referencias_historicas']).T[['auc_pr','precision','recall','f1','costo_q']]

,auc_pr,precision,recall,f1,costo_q
V1,0.428504,0.133923,0.7217,0.225923,6193620.0
V2,0.453579,0.148822,0.702887,0.245636,6078120.0
V3,0.512502,0.190496,0.726889,0.301879,5250540.0


<div class="section"><h2>8 · Capacidad y segmentos</h2></div>

Top-K conecta el ranking con cupos reales de analistas. Los segmentos fueron recalculados con el candidato y umbral recomendados; no describen el ensamble descartado.

In [9]:
pd.DataFrame(R['metricas_top_k'])

,tasa_revision,k,precision_at_k,recall_at_k
0,0.001,88,1.000000,0.028544
1,0.005,442,0.927602,0.132987
2,0.010,885,0.882486,0.253325
3,0.020,1771,0.675325,0.387934
4,0.050,4429,0.386543,0.555303


<div class="section"><h2>9 · Conclusión y límites</h2></div>

La mejora proviene principalmente de más cobertura, categorías nativas, ausencia preservada y ponderación de recencia. La GRU no se amplió porque la falsificación V1 mostró señal de orden débil. Antes de producción se necesita cohorte nueva, identidad confiable, costos reales, privacidad, equidad, seguridad y monitoreo.

In [10]:
R['limitaciones']

['Benchmark reutilizado y no ciego',
 'Identidad aproximada',
 'Costos académicos',
 'La GRU V1 no mostró señal de orden suficiente; no se fabricó un stacking GRU sin predicciones OOF',
 'Una cohorte nueva es necesaria para confirmación']

<div class="section"><h2>Referencias APA 7</h2></div>

IEEE Computational Intelligence Society. (2019). *IEEE-CIS Fraud Detection* [Conjunto de datos]. Kaggle. https://www.kaggle.com/competitions/ieee-fraud-detection

Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems, 30*.

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

**Declaración de IA.** Se utilizó asistencia para código, redacción y auditoría. Los autores ejecutaron y verificaron el experimento y asumen responsabilidad por sus conclusiones.